# Peer University Benchmarking Analysis

**Objective:** Compare AUB citation patterns against peer institutions (Lehigh, Marquette, Villanova)

## Data Sources:
- AUB: `data/processed/cleaned_data.csv` (already cleaned)
- Lehigh: `data/raw/lehigh.csv`
- Marquette: `data/raw/marquette.csv`
- Villanova: `data/raw/villanova.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

%matplotlib inline

## 1. Load All Data Sources

In [ ]:
# Load AUB data (already cleaned)
aub = pd.read_csv('../data/processed/cleaned_data.csv')
print(f"AUB: {len(aub)} publications")
print(f"Columns: {list(aub.columns)}")
aub.head()

In [ ]:
# DIAGNOSTIC: Load peer universities and inspect structure
# These CSVs appear to have unusual formatting - let's investigate

print("=" * 60)
print("LOADING PEER UNIVERSITY DATA")
print("=" * 60)

# Try loading with default settings first
lehigh_raw = pd.read_csv('../data/raw/lehigh.csv')
marquette_raw = pd.read_csv('../data/raw/marquette.csv')
villanova_raw = pd.read_csv('../data/raw/villanova.csv')

print(f"\nLehigh shape: {lehigh_raw.shape}")
print(f"Marquette shape: {marquette_raw.shape}")
print(f"Villanova shape: {villanova_raw.shape}")

print("\n" + "=" * 60)
print("FIRST 5 ROWS OF LEHIGH (to understand structure):")
print("=" * 60)
print(lehigh_raw.head())

print("\n" + "=" * 60)
print("LEHIGH COLUMNS:")
print("=" * 60)
print(list(lehigh_raw.columns))

## 2. Parse Unusual CSV Format

Based on the inspection above, these CSVs likely have:
- Headers in the first column instead of first row (transposed)
- Or metadata rows at the top that need to be skipped
- Or data exported from Excel with unusual formatting

**ACTION REQUIRED:** Examine the output above and adjust the parsing logic below

In [ ]:
# OPTION 1: If data is TRANSPOSED (headers in first column)
# Uncomment and run if needed:
# lehigh = lehigh_raw.T
# lehigh.columns = lehigh.iloc[0]  # First row becomes column names
# lehigh = lehigh.drop(lehigh.index[0]).reset_index(drop=True)

# OPTION 2: If there are HEADER ROWS to skip
# Uncomment and adjust skip_rows as needed:
# lehigh = pd.read_csv('../data/raw/lehigh.csv', skiprows=2)
# marquette = pd.read_csv('../data/raw/marquette.csv', skiprows=2)
# villanova = pd.read_csv('../data/raw/villanova.csv', skiprows=2)

# OPTION 3: If it's an Excel export, try reading as Excel
# Uncomment if CSVs are actually .xlsx:
# lehigh = pd.read_excel('../data/raw/lehigh.xlsx')
# marquette = pd.read_excel('../data/raw/marquette.xlsx')
# villanova = pd.read_excel('../data/raw/villanova.xlsx')

# For now, keep the raw versions
lehigh = lehigh_raw.copy()
marquette = marquette_raw.copy()
villanova = villanova_raw.copy()

print("Current structure after parsing:")
print(f"Lehigh: {lehigh.shape}")
print(f"Columns: {list(lehigh.columns)[:5]} ... (showing first 5)")
lehigh.head(3)

In [ ]:
## 3. Compare Schemas

Now compare the peer university structure with AUB's cleaned data

print("AUB Columns:")
print(list(aub.columns))
print(f"\nAUB Sample:")
display(aub.head(2))

print("\n" + "=" * 60)
print("Peer Universities - Current Columns:")
print("=" * 60)
print(f"\nLehigh: {list(lehigh.columns)[:10]} ... ({len(lehigh.columns)} total)")
print(f"Marquette: {list(marquette.columns)[:10]} ... ({len(marquette.columns)} total)")
print(f"Villanova: {list(villanova.columns)[:10]} ... ({len(villanova.columns)} total)")

print("\n" + "=" * 60)
print("NEXT STEP:")
print("=" * 60)
print("Based on the structure above, you need to:")
print("1. Understand how the peer CSVs are formatted")
print("2. Transform them to match AUB's structure (one row per publication)")
print("3. Map/rename columns to match AUB's schema")

In [ ]:
## 4. Standardize Column Names

Map peer university columns to match AUB schema

**IMPORTANT:** Adjust this mapping based on the actual column names you found above

# Example column mapping (ADJUST BASED ON YOUR DATA!)
# 
# If peer CSVs have standard columns like:
# column_mapping = {
#     'Title': 'title',
#     'Year': 'year', 
#     'Times Cited': 'citations',
#     'Source Title': 'venue',
#     'Authors': 'authors',
#     # Add more as needed
# }
#
# lehigh = lehigh.rename(columns=column_mapping)
# marquette = marquette.rename(columns=column_mapping)
# villanova = villanova.rename(columns=column_mapping)

# Placeholder - uncomment and adjust above when ready
print("⚠️  Column mapping not yet applied - adjust the code above first!")
print(f"\nCurrent lehigh columns: {list(lehigh.columns)[:5]}...")
print(f"Target AUB columns: {list(aub.columns)}")

In [ ]:
## 5. Add Institution Tags & Merge

**ONLY RUN THIS AFTER** columns are properly mapped above!

# Add institution column to each dataset
aub_tagged = aub.copy()
aub_tagged['institution'] = 'AUB'

lehigh_tagged = lehigh.copy()
lehigh_tagged['institution'] = 'Lehigh'

marquette_tagged = marquette.copy()
marquette_tagged['institution'] = 'Marquette'

villanova_tagged = villanova.copy()
villanova_tagged['institution'] = 'Villanova'

# Find common columns across all datasets
common_cols = list(set(aub_tagged.columns) & 
                   set(lehigh_tagged.columns) & 
                   set(marquette_tagged.columns) & 
                   set(villanova_tagged.columns))

print(f"Common columns found: {sorted(common_cols)}")

# Select only common columns for merging
all_universities = pd.concat([
    aub_tagged[common_cols],
    lehigh_tagged[common_cols],
    marquette_tagged[common_cols],
    villanova_tagged[common_cols]
], ignore_index=True)

print(f"\n✅ Merged dataset created!")
print(f"Total publications: {len(all_universities):,}")
print(f"\nBreakdown by institution:")
print(all_universities['institution'].value_counts())

In [ ]:
## 6. Save Merged Dataset

# Save merged dataset
output_path = '../data/processed/all_universities.parquet'
all_universities.to_parquet(output_path, index=False)
print(f"✅ Saved to: {output_path}")

# Also save as CSV for easier inspection
csv_path = '../data/processed/all_universities.csv'
all_universities.to_csv(csv_path, index=False)
print(f"✅ Also saved CSV: {csv_path}")

print(f"\nDataset info:")
print(f"  - Shape: {all_universities.shape}")
print(f"  - Columns: {list(all_universities.columns)}")

In [ ]:
## 7. Comparative Analysis

### 7.1 Citation Statistics

In [ ]:
# Visualization: Citation distribution comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box plot
ax1 = axes[0, 0]
all_universities.boxplot(column='citations', by='institution', ax=ax1)
ax1.set_title('Citation Distribution by Institution')
ax1.set_xlabel('Institution')
ax1.set_ylabel('Citations')
plt.sca(ax1)
plt.xticks(rotation=45)

# Violin plot
ax2 = axes[0, 1]
sns.violinplot(data=all_universities, x='institution', y='citations', ax=ax2)
ax2.set_title('Citation Density by Institution')
ax2.set_xlabel('Institution')
ax2.set_ylabel('Citations')
plt.sca(ax2)
plt.xticks(rotation=45)

# Bar chart - mean citations
ax3 = axes[1, 0]
citation_stats['mean'].plot(kind='bar', ax=ax3, color='skyblue')
ax3.set_title('Mean Citations by Institution')
ax3.set_xlabel('Institution')
ax3.set_ylabel('Mean Citations')
plt.sca(ax3)
plt.xticks(rotation=45)

# Publication counts
ax4 = axes[1, 1]
all_universities['institution'].value_counts().plot(kind='bar', ax=ax4, color='coral')
ax4.set_title('Number of Publications by Institution')
ax4.set_xlabel('Institution')
ax4.set_ylabel('Count')
plt.sca(ax4)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

### 7.2 Temporal Trends

In [ ]:
# Publications over time
yearly_pubs = all_universities.groupby(['institution', 'year']).size().reset_index(name='count')

plt.figure(figsize=(14, 6))
for inst in all_universities['institution'].unique():
    data = yearly_pubs[yearly_pubs['institution'] == inst]
    plt.plot(data['year'], data['count'], marker='o', label=inst)

plt.title('Publication Trends by Institution')
plt.xlabel('Year')
plt.ylabel('Number of Publications')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Average citations over time
yearly_citations = all_universities.groupby(['institution', 'year'])['citations'].mean().reset_index()

plt.figure(figsize=(14, 6))
for inst in all_universities['institution'].unique():
    data = yearly_citations[yearly_citations['institution'] == inst]
    plt.plot(data['year'], data['citations'], marker='o', label=inst)

plt.title('Average Citations per Publication by Institution Over Time')
plt.xlabel('Year')
plt.ylabel('Average Citations')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 7.3 High-Impact Publications

In [ ]:
# Define thresholds for high-impact papers
thresholds = [10, 25, 50, 100]

high_impact = pd.DataFrame()
for threshold in thresholds:
    counts = all_universities[all_universities['citations'] >= threshold].groupby('institution').size()
    high_impact[f'{threshold}+ citations'] = counts

high_impact = high_impact.fillna(0).astype(int)
high_impact

In [ ]:
# Percentage of high-impact papers
total_pubs = all_universities['institution'].value_counts()

high_impact_pct = pd.DataFrame()
for col in high_impact.columns:
    high_impact_pct[col] = (high_impact[col] / total_pubs * 100).round(2)

high_impact_pct

In [ ]:
# Visualization
high_impact_pct.plot(kind='bar', figsize=(12, 6))
plt.title('Percentage of High-Impact Publications by Institution')
plt.xlabel('Institution')
plt.ylabel('Percentage (%)')
plt.legend(title='Citation Threshold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Key Findings Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS: AUB vs Peer Universities")
print("=" * 60)
print(f"\nTotal Publications:")
print(all_universities['institution'].value_counts())
print(f"\nCitation Statistics:")
print(citation_stats[['mean', 'median', 'max']])
print(f"\nHigh-Impact Papers (100+ citations):")
print(high_impact['100+ citations'])
print(f"\nPercentage High-Impact:")
print(high_impact_pct['100+ citations'])

## 9. Next Steps

1. **Field/discipline analysis** - Compare citation patterns across research areas
2. **Author-level metrics** - h-index, productivity comparisons
3. **Venue quality** - Journal impact factors, conference rankings
4. **Collaboration patterns** - Co-authorship networks
5. **Geographic analysis** - International collaboration patterns